# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdarshIsaac/NewRepoML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract answer 1:** One row represents one pseudonymized **client-content pair at the March 2026 month-end snapshot**. I will use `fact_content_daily_performance` and aggregate daily records by `client_hash_id` and `content_hash_id`. Features are measured in the preceding 30 days (2026-02-01 through 2026-02-28), the decision moment is 2026-03-01, and the observed label is March performance (2026-03-01 through 2026-03-31). This supports ranking pages for review based on information available before March.

In [ ]:
%pip -q install duckdb huggingface_hub pandas

import os
import getpass
from pathlib import Path

import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Enter your Hugging Face READ token (input hidden): "
)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

FEATURE_START = "2026-02-01"
FEATURE_END = "2026-02-28"
LABEL_START = "2026-03-01"
LABEL_END = "2026-03-31"

print("Warehouse connection configured")
print("Feature window:", FEATURE_START, "to", FEATURE_END)
print("Label window:", LABEL_START, "to", LABEL_END)

Note: you may need to restart the kernel to use updated packages.


## 2. Fields: feature / label / context / excluded

**Contract answer 2:**

- **Features (five maximum):** `prev30_impressions`, `prev30_clicks`, `prev30_sessions`, `prev30_avg_position`, and `prev30_active_days`. Each is available by the decision moment and describes the page's preceding 30-day search/analytics activity.
- **Label / proxy:** `march_impressions`, used to derive the observed March outcome `march_decline_label` by comparing March impressions with the February feature window. This is an observed outcome, not a causal claim.
- **Context:** `client_hash_id`, `content_hash_id`, and `report_date` for grouping, joining, filtering, and client-level holdout splits. They are never model features.
- **Excluded:** `fact_content_query_90d`, because its fixed recent 90-day window overlaps future periods relative to this March decision; all post-2026-03-01 fields, because they are unavailable at the decision moment; and IDs, because pseudonyms would encourage memorization rather than generalization.

In [4]:
FEATURE_COLUMNS = [
    "prev30_impressions",
    "prev30_clicks",
    "prev30_sessions",
    "prev30_avg_position",
    "prev30_active_days",
]
CONTEXT_COLUMNS = ["client_hash_id", "content_hash_id"]
LABEL_COLUMN = "march_impressions"

assert len(FEATURE_COLUMNS) == 5
assert set(FEATURE_COLUMNS).isdisjoint(CONTEXT_COLUMNS)
assert LABEL_COLUMN not in FEATURE_COLUMNS

print("Features (5):", FEATURE_COLUMNS)
print("Label/proxy:", LABEL_COLUMN, "with decline derived from the March window")
print("Context only:", CONTEXT_COLUMNS)
print("Excluded: query-level table, future fields, and identifiers as predictors")

Features (5): ['prev30_impressions', 'prev30_clicks', 'prev30_sessions', 'prev30_avg_position', 'prev30_active_days']
Label/proxy: march_impressions with decline derived from the March window
Context only: ['client_hash_id', 'content_hash_id']
Excluded: query-level table, future fields, and identifiers as predictors


## 3. Verify it with queries (grain, counts, missing values, windows)

**Contract answer 3:** The following three small queries verify the claimed monthly slice. Query 1 tests that daily grain has no duplicate client-page-date records. Query 2 checks the selected dates and row count. Query 3 uses `IS TRUE` for measurement availability and reports how many records remain for an honest feature frame.

In [2]:
# Query 1: grain check. No rows should be returned.
duplicate_grain = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM {FACT}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{LABEL_END}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

# Query 2: selected slice count and date span.
slice_summary = con.sql(f"""
    SELECT COUNT(*) AS daily_rows,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {FACT}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{LABEL_END}'
""").df()

# Query 3: availability check. NULL is intentionally not treated as TRUE.
availability_summary = con.sql(f"""
    SELECT
        COUNT(*) AS rows_in_two_months,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {FACT}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{LABEL_END}'
""").df()

assert duplicate_grain.empty, "Daily fact grain has duplicate client-content-date rows."
assert slice_summary.loc[0, "first_date"] == pd.Timestamp(FEATURE_START)
assert slice_summary.loc[0, "last_date"] == pd.Timestamp(LABEL_END)
assert availability_summary.loc[0, "both_available_rows"] > 0

print("Query 1 - duplicate grain rows:", len(duplicate_grain))
print("Query 2 - slice summary:")
display(slice_summary)
print("Query 3 - availability summary:")
display(availability_summary)

Query 1 - duplicate grain rows: 0
Query 2 - slice summary:


,daily_rows,clients,content_items,first_date,last_date
0,17196486,59,349411,2026-02-01,2026-03-31


Query 3 - availability summary:


,rows_in_two_months,gsc_available_rows,ga4_available_rows,both_available_rows
0,17196486,6232844,559287,489027


## 4. Data limits

**Contract answer 4:** This contract cannot establish causality, explain Google ranking decisions, or guarantee that a March decline continues. The warehouse is an unbalanced panel: clients have different history starts and some rows lack usable GSC or GA4 measurement. The selected frame therefore filters to rows where both availability flags are `TRUE`, uses a fixed mid-panel window only as a measured example, and treats missingness as a limitation rather than zero performance.

**Output:** A five-feature client-page frame and an observed March decline label that can support a human review ranking after client-held-out validation.

In [5]:
# Build the contract's small feature frame in DuckDB.
feature_frame = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
                 THEN gsc_impressions ELSE 0 END) AS prev30_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
                 THEN gsc_clicks ELSE 0 END) AS prev30_clicks,
        SUM(CASE WHEN report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
                 AND ga4_data_available IS TRUE
                 THEN ga4_sessions ELSE 0 END) AS prev30_sessions,
        AVG(CASE WHEN report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
                 AND gsc_avg_position > 0
                 THEN gsc_avg_position END) AS prev30_avg_position,
        COUNT(*) FILTER (WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
                         AND gsc_impressions > 0) AS prev30_active_days,
        SUM(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
                 THEN gsc_impressions ELSE 0 END) AS march_impressions
    FROM {FACT}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{LABEL_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT *,
       (march_impressions < 0.8 * prev30_impressions)::INTEGER AS march_decline_label
FROM monthly
WHERE prev30_impressions > 0
""").df()

assert set(FEATURE_COLUMNS).issubset(feature_frame.columns)
assert feature_frame[CONTEXT_COLUMNS].drop_duplicates().shape[0] == len(feature_frame)
assert feature_frame[FEATURE_COLUMNS].nunique(dropna=False).gt(1).all()

print("Feature frame shape:", feature_frame.shape)
print("One row = one client-content monthly snapshot:", len(feature_frame) == feature_frame[CONTEXT_COLUMNS].drop_duplicates().shape[0])
print("Observed March decline rate:", round(feature_frame["march_decline_label"].mean() * 100, 1), "%")
print("Feature meanings:")
for column in FEATURE_COLUMNS:
    print(f"- {column}: knowable before the 2026-03-01 decision moment")
display(feature_frame[CONTEXT_COLUMNS + FEATURE_COLUMNS + ["march_impressions", "march_decline_label"]].head())

Feature frame shape: (153559, 9)
One row = one client-content monthly snapshot: True
Observed March decline rate: 30.1 %
Feature meanings:
- prev30_impressions: knowable before the 2026-03-01 decision moment
- prev30_clicks: knowable before the 2026-03-01 decision moment
- prev30_sessions: knowable before the 2026-03-01 decision moment
- prev30_avg_position: knowable before the 2026-03-01 decision moment
- prev30_active_days: knowable before the 2026-03-01 decision moment


,client_hash_id,content_hash_id,prev30_impressions,prev30_clicks,prev30_sessions,prev30_avg_position,prev30_active_days,march_impressions,march_decline_label
0,client_62f4a7e64f5e0096,content_bc5af5c05fd34858,128.0,0.0,0.0,17.007797,27,44.0,1
1,client_62f4a7e64f5e0096,content_1f01eeb074daf451,294.0,1.0,0.0,3.958943,28,225.0,1
2,client_62f4a7e64f5e0096,content_6f5b57f67fb8262b,837.0,2.0,0.0,6.061206,28,1258.0,0
3,client_62f4a7e64f5e0096,content_3e296e654749d63d,1118.0,1.0,0.0,4.127025,28,1187.0,0
4,client_62f4a7e64f5e0096,content_3baecba845fb045b,186.0,0.0,0.0,15.846745,27,76.0,1


## 5. Leakage trap

Intentionally adding March outcome information to the ranking score creates leakage. The leaky score below uses `march_impressions`, which is unavailable at the 2026-03-01 decision moment, so it is only a demonstration and must not be retained as a feature.

In [7]:
leaky_rank = feature_frame.assign(leaky_score=-feature_frame["march_impressions"])
leaky_precision_at_50 = feature_frame.loc[
    leaky_rank.nlargest(50, "leaky_score").index, "march_decline_label"
].mean()

print("Intentional leaky Precision@50:", round(leaky_precision_at_50, 3))
print("Honest contract action: delete march_impressions and march_decline_label from the feature list; keep them only as outcome fields.")
assert "march_impressions" not in FEATURE_COLUMNS
assert "march_decline_label" not in FEATURE_COLUMNS

Intentional leaky Precision@50: 1.0
Honest contract action: delete march_impressions and march_decline_label from the feature list; keep them only as outcome fields.


## Self-check

- [x] Five contract answers are written in plain words.
- [x] Three verification queries cover grain, count/date span, and availability with `IS TRUE`.
- [x] A five-feature frame is built from the same mid-panel month and each feature is explained.
- [x] One label-derived leakage field is intentionally demonstrated, then excluded from `FEATURE_COLUMNS`.
- [ ] Run the notebook top to bottom after entering your Hugging Face READ token, then commit it under `work/notebooks/`.